In [10]:
from pathlib import Path

log_path = Path("backfill_log.txt").resolve()

with open(log_path, "r", errors="replace") as f:
    lines = f.readlines()

size_bytes = log_path.stat().st_size
num_lines = len(lines)

print(f"Path: {log_path}")
print(f"Size: {size_bytes:,} bytes ({size_bytes / 1024 / 1024:.2f} MB)")
print(f"Number of lines: {num_lines:,}")
print()

Path: /Users/anil/code/reroll-data/backfill_log.txt
Size: 3,743,320 bytes (3.57 MB)
Number of lines: 43,916


In [11]:
import re
from dataclasses import dataclass
from pprint import pprint

full_text = "".join(lines)

# Split on the delimiter, keeping the delimiter attached to the start of each
# following chunk (lookahead split).
raw_chunks = re.split(r"(?=! metadata parse failed for)", full_text)

# Drop the leading chunk before the first delimiter (e.g. the invocation line),
# and strip trailing whitespace from each entry.
buckets = [
    chunk.strip()
    for chunk in raw_chunks
    if chunk.strip().startswith("! metadata parse failed for")
]

print(f"Number of buckets: {len(buckets):,}")


@dataclass
class ParsedFailure:
    metadata_id: str | None
    fields: list[str]
    sub_types: list[str]
    raw: str
    type: str = "validation_error"


_ID_RE = re.compile(r"^! metadata parse failed for ([^:]+):")
_VALIDATION_RE = re.compile(
    r"^! metadata parse failed for ([^:]+): ValidationError: \d+ validation errors? for \w+\n(.*)",
    re.DOTALL,
)
# Field name lines are unindented, immediately followed by an indented detail
# line, e.g. "requires_dist\n  Value error, ...".
_FIELD_DETAIL_RE = re.compile(r"^([A-Za-z_][A-Za-z0-9_]*)\n  (.+)$", re.MULTILINE)
# Strip the trailing pydantic `[type=..., input_value=..., ...]` annotation.
_BRACKET_SUFFIX_RE = re.compile(r"\s*\[type=.*$")

# Ordered (most specific first) rules mapping a detail message to a sub_type.
_SUB_TYPE_RULES = [
    (
        "mismatched_parenthesis",
        r"^Value error, Expected matching RIGHT_PARENTHESIS for LEFT_PARENTHESIS",
    ),
    ("expected_package_name", r"^Value error, Expected package name at the start"),
    (
        "expected_marker_or_string",
        r"^Value error, Expected a marker variable or quoted string",
    ),
    ("expected_end_or_semicolon", r"^Value error, Expected end or semicolon"),
    ("invalid_specifier", r"^Value error, Invalid specifier:"),
    (
        "local_version_label_misuse",
        r"^Value error, Local version label can only be used",
    ),
    ("wildcard_suffix_misuse", r"^Value error, \.\* suffix can only be used"),
    ("unknown_license", r"^Value error, Unknown license:"),
    ("invalid_license_expression", r"^Value error, Invalid license expression:"),
    ("invalid_version", r"^Value error, Invalid version:"),
    ("invalid_package_name", r"^Value error, name is invalid:"),
    ("invalid_string", r"^Input should be a valid string"),
    ("undecodable_body", r"^undecodable body:"),
]


def classify_detail(detail: str) -> str:
    cleaned = _BRACKET_SUFFIX_RE.sub("", detail).strip()
    for sub_type, pattern in _SUB_TYPE_RULES:
        if re.match(pattern, cleaned):
            return sub_type
    return "other"


def parse_entry(raw: str) -> ParsedFailure:
    match = _VALIDATION_RE.match(raw)
    if match:
        metadata_id, body = match.groups()
        field_detail_pairs = _FIELD_DETAIL_RE.findall(body)
        return ParsedFailure(
            metadata_id=metadata_id,
            fields=[field for field, _ in field_detail_pairs],
            sub_types=[classify_detail(detail) for _, detail in field_detail_pairs],
            raw=raw,
            type="validation_error",
        )

    # Doesn't match the expected ValidationError pattern (e.g. undecodable
    # body errors) -- still try to pull out the metadata id, mark unknown.
    id_match = _ID_RE.match(raw)
    rest = raw[id_match.end() :].strip() if id_match else raw
    return ParsedFailure(
        metadata_id=id_match.group(1) if id_match else None,
        fields=[],
        sub_types=[classify_detail(rest)],
        raw=raw,
        type="unknown",
    )


parsed_failures = [parse_entry(b) for b in buckets]

num_unknown = sum(p.type == "unknown" for p in parsed_failures)
num_other = sum("other" in p.sub_types for p in parsed_failures)
print(
    f"Parsed entries: {len(parsed_failures):,} ({num_unknown} unknown type, {num_other} 'other' sub_type)"
)
print()
pprint(parsed_failures[:2])

Number of buckets: 8,056
Parsed entries: 8,056 (24 unknown type, 0 'other' sub_type)

[ParsedFailure(metadata_id='a9903d24c3b1ed1d0b82635637c4ea006a33d394dbc91896371a330f34d5a49e',
               fields=['requires_dist'],
               sub_types=['mismatched_parenthesis'],
               raw='! metadata parse failed for '
                   'a9903d24c3b1ed1d0b82635637c4ea006a33d394dbc91896371a330f34d5a49e: '
                   'ValidationError: 1 validation error for WheelMetadata\n'
                   'requires_dist\n'
                   '  Value error, Expected matching RIGHT_PARENTHESIS for '
                   'LEFT_PARENTHESIS, after version specifier\n'
                   '    youtube-dl (>=2020.11.12aiohttp>=3.6.0)\n'
                   '               ~~~~~~~~~~~~~~^ [type=value_error, '
                   "input_value=('youtube-dl (>=2020.11.12aiohttp>=3.6.0)',), "
                   'input_type=tuple]\n'
                   '    For further information visit '
               

In [14]:
from collections import Counter

# Capture the FULL multi-line error block per field (not just the first
# detail line), so we can tell a literal duplicate error (e.g. the
# youtube-dl parenthesis bug repeating verbatim) apart from many distinct
# specifier values that just happen to share a sub_type.
_FIELD_BLOCK_RE = re.compile(
    r"^([A-Za-z_][A-Za-z0-9_]*)\n((?:(?!^[A-Za-z_][A-Za-z0-9_]*$).*\n?)+)",
    re.MULTILINE,
)


def field_blocks(raw: str) -> list[tuple[str, str]]:
    match = _VALIDATION_RE.match(raw)
    if not match:
        return []
    body = match.group(2)
    return [(field, block.strip()) for field, block in _FIELD_BLOCK_RE.findall(body)]


# One record per (metadata_id, field, sub_type) failure.
records = []
for p in parsed_failures:
    if p.type == "validation_error":
        for (field, block), sub_type in zip(field_blocks(p.raw), p.sub_types):
            records.append(
                {
                    "metadata_id": p.metadata_id,
                    "field": field,
                    "sub_type": sub_type,
                    "block": block,
                }
            )
    else:
        id_match = _ID_RE.match(p.raw)
        rest = p.raw[id_match.end() :].strip() if id_match else p.raw
        records.append(
            {
                "metadata_id": p.metadata_id,
                "field": None,
                "sub_type": p.sub_types[0],
                "block": rest,
            }
        )

print(f"Total field-level failure records: {len(records):,}")
print()

sub_type_counts = Counter(r["sub_type"] for r in records)
print("Counts by sub_type:")
for sub_type, count in sub_type_counts.most_common():
    print(f"  {count:6,}  {sub_type}")

print()
print("Counts by field:")
for field, count in Counter(r["field"] for r in records).most_common():
    print(f"  {count:6,}  {field}")

print()
print("=" * 78)
print("Duplicate-string vs. distinct-variant breakdown per sub_type")
print("(is the count inflated by ONE repeated literal error, or many")
print(" different specifier/version/license/etc. values?)")
print("=" * 78)
for sub_type, total in sub_type_counts.most_common():
    block_counter = Counter(r["block"] for r in records if r["sub_type"] == sub_type)
    unique_blocks = len(block_counter)
    top_block, top_count = block_counter.most_common(1)[0]
    top_share = top_count / total
    verdict = (
        "DOMINATED BY ONE STRING" if top_share >= 0.5 else "MANY DISTINCT VARIANTS"
    )
    dup_ratio = total / unique_blocks
    print(
        f"\n[{sub_type}]  total={total:,}  unique_variants={unique_blocks:,}  "
        f"avg_dup={dup_ratio:.1f}x  top_variant={top_count:,} ({top_share:.0%})  -> {verdict}"
    )
    print(f"  top variant: {top_block[:160]!r}")

print()
print("=" * 78)
print("Top 5 variants within each of the 3 largest sub_types")
print("(shows the real spread driving each category's total)")
print("=" * 78)
for sub_type, total in sub_type_counts.most_common(3):
    block_counter = Counter(r["block"] for r in records if r["sub_type"] == sub_type)
    print(f"\n[{sub_type}]  total={total:,}  unique_variants={len(block_counter):,}")
    for block, count in block_counter.most_common(5):
        # Show line 1 (the generic message) + line 2 (the actual offending
        # value) since for some sub_types the real variety is on line 2.
        preview_lines = block.splitlines()[:2]
        preview = " | ".join(l.strip() for l in preview_lines)
        print(f"    {count:5,} ({count / total:5.1%})  {preview[:130]}")

print()
print("NOTE: for invalid_string, the differing 'input_value=<object object")
print("at 0x...>' addresses are NOT distinct data -- they're the same")
print("sentinel/None-like value with a different Python object id each run,")
print("so that category is effectively ONE root cause, not 13 real variants.")

Total field-level failure records: 8,063

Counts by sub_type:
   5,453  mismatched_parenthesis
   1,123  invalid_specifier
     649  invalid_string
     240  local_version_label_misuse
     198  unknown_license
     159  wildcard_suffix_misuse
     139  invalid_license_expression
      48  invalid_version
      24  undecodable_body
      12  expected_end_or_semicolon
      11  invalid_package_name
       5  expected_package_name
       2  expected_marker_or_string

Counts by field:
   5,871  requires_dist
   1,123  requires_python
     660  name
     337  license_expression
      48  version
      24  None

Duplicate-string vs. distinct-variant breakdown per sub_type
(is the count inflated by ONE repeated literal error, or many
 different specifier/version/license/etc. values?)

[mismatched_parenthesis]  total=5,453  unique_variants=1,602  avg_dup=3.4x  top_variant=93 (2%)  -> MANY DISTINCT VARIANTS
  top variant: 'Value error, Expected matching RIGHT_PARENTHESIS for LEFT_PARENTHESIS, 

In [17]:
## SCRATCH: metadata_ids + unique blocks for invalid_version and invalid_string
print("=== invalid_version: (metadata_id, block) pairs ===")
for r in invalid_version_records:
    print(r["metadata_id"], "|", r["block"][:90])

print()
print(
    "=== invalid_string: unique blocks with counts and one sample metadata_id each ==="
)
from collections import Counter, defaultdict

block_to_ids = defaultdict(list)
for r in invalid_string_records:
    block_to_ids[r["block"]].append(r["metadata_id"])

for block, ids in sorted(block_to_ids.items(), key=lambda kv: -len(kv[1])):
    print(len(ids), "|", ids[0], "|", block[:80])

=== invalid_version: (metadata_id, block) pairs ===
bb04d31320a1551d0cc8095267733512a27462ff2862dd3367bcbd48cb47c507 | Value error, Invalid version: '0.2.1.Perceval' [type=value_er
ror, input_value='0.2.1.Perce
84231e2a235af8e894647cf8549b28fbc0b1616ae7e208dbcfacd6d2ed831049 | Value error, Invalid version: '0.1.3fix1' [type=value_error,
input_value='0.1.3fix1', inpu
e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855 | Value error, Invalid version: 'None' [type=value_error, input
_value=None, input_type=NoneT
99f75bf9a34c14d55871622ebd2fd6674a99990ecd191d3e0fe584ad9e586898 | Value error, Invalid version: '1.5.3.final.0' [type=value_err
or, input_value='1.5.3.final.
64837ac130b0100ecd192e650e069251fe567310383ac72c3ad802710d725871 | Value error, Invalid version: '1.6.1.final.0' [type=value_err
or, input_value='1.6.1.final.
36cff563a589c23f195fb8e557af343c808b4f05489be3dab2704eb1658ab026 | Value error, Invalid version: '0.0.1-git.4.842fbc8' [type=val
ue_error, input_value='0.